# Giai đoạn 5: Mô hình Học Máy — K-Means Clustering & Regression
**Dự án:** Phân tích dữ liệu thương mại điện tử Olist (Brazil, 2016–2018)

**Thành viên thực hiện:** Nhóm ADY291c-AI2107-Group2

**Đầu vào:** `data/olist.db` — view `v_customer_rfm` & `view_order_clean`

---

## Mục tiêu
| Mô hình | Bài toán | Đầu vào | Đầu ra |
|---|---|---|---|
| **K-Means Clustering** | Phân khúc khách hàng | RFM (Recency, Frequency, Monetary) | Nhãn phân khúc (0–3) |
| **Linear Regression** | Dự đoán review score | Đặc trưng đơn hàng | `review_score` (1–5) |

---

## 0. Import thư viện & kết nối Database

In [3]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Kết nối DB
DB_PATH = '../data/olist.db'
conn = sqlite3.connect(DB_PATH)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
PALETTE = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']

print(f'Ket noi thanh cong: {DB_PATH}')

Ket noi thanh cong: ../data/olist.db


---
# PHẦN 1: K-MEANS CLUSTERING — Phân khúc khách hàng theo RFM

**RFM Framework:**
- **R**ecency: Bao lâu rồi khách chưa mua? (ngày kể từ lần mua cuối)
- **F**requency: Khách mua bao nhiêu lần?
- **M**onetary: Khách chi bao nhiêu tiền?


## 1.1 Chuẩn bị dữ liệu RFM

In [4]:
rfm = pd.read_sql_query("""
    SELECT
        customer_unique_id,
        customer_state,
        CAST(
            julianday('2018-10-18') -
            julianday(MAX(order_purchase_timestamp))
        AS INTEGER) AS recency_days,
        COUNT(DISTINCT order_id)     AS frequency,
        ROUND(SUM(payment_value), 2) AS monetary
    FROM view_order_clean
    GROUP BY customer_unique_id, customer_state
""", conn)

print(f'So luong khach hang duy nhat: {len(rfm):,}')
print()
print(rfm[['recency_days','frequency','monetary']].describe().round(2))

DatabaseError: Execution failed on sql '
    SELECT
        customer_unique_id,
        customer_state,
        CAST(
            julianday('2018-10-18') -
            julianday(MAX(order_purchase_timestamp))
        AS INTEGER) AS recency_days,
        COUNT(DISTINCT order_id)     AS frequency,
        ROUND(SUM(payment_value), 2) AS monetary
    FROM view_order_clean
    GROUP BY customer_unique_id, customer_state
': no such table: view_order_clean

## 1.2 Chuẩn hóa dữ liệu (StandardScaler)

In [ ]:
# Loại bỏ outlier Monetary cực đoan (99.5th percentile)
cap = rfm['monetary'].quantile(0.995)
rfm_clean = rfm[rfm['monetary'] <= cap].copy()
print(f'Sau khi cap outlier: {len(rfm_clean):,} khach hang (bo {len(rfm)-len(rfm_clean):,} dong)')

features = ['recency_days', 'frequency', 'monetary']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_clean[features])

print(f'\nDu lieu da chuan hoa — shape: {X_scaled.shape}')

## 1.3 Chọn số cụm tối ưu — Elbow Method & Silhouette Score

In [ ]:
K_RANGE = range(2, 9)
inertias, silhouettes = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels, sample_size=5000, random_state=42))
    print(f'  k={k}  Inertia={km.inertia_:,.0f}  Silhouette={silhouettes[-1]:.4f}')

# Vẽ Elbow + Silhouette
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_RANGE, inertias, 'o-', color=PALETTE[0], linewidth=2)
axes[0].set_title('Elbow Method — Inertia')
axes[0].set_xlabel('So cum k')
axes[0].set_ylabel('Inertia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_RANGE, silhouettes, 's-', color=PALETTE[1], linewidth=2)
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('So cum k')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

best_k = list(K_RANGE)[silhouettes.index(max(silhouettes))]
axes[1].axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
axes[1].legend()

plt.suptitle('Chon so cum toi uu cho K-Means', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/kmeans_elbow.png', bbox_inches='tight')
plt.show()
print(f'\n=> Best k = {best_k} (Silhouette = {max(silhouettes):.4f})')

## 1.4 Huấn luyện K-Means với k=4 (phân khúc thực tế)

In [ ]:
K_FINAL = 4  # 4 phân khúc kinh doanh: Loyal, At Risk, New, Champion

km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20)
rfm_clean['cluster'] = km_final.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, rfm_clean['cluster'], sample_size=5000, random_state=42)
print(f'K-Means k={K_FINAL} | Inertia={km_final.inertia_:,.0f} | Silhouette={sil:.4f}')

# Profile từng cluster
profile = rfm_clean.groupby('cluster')[features].mean().round(2)
profile['count'] = rfm_clean.groupby('cluster').size()
profile['pct'] = (profile['count'] / len(rfm_clean) * 100).round(1)

# Gán nhãn kinh doanh
# Sắp xếp theo monetary DESC để gán nhãn nhất quán
profile = profile.sort_values('monetary', ascending=False)
labels_biz = ['Champion', 'Loyal', 'At Risk', 'New/Occasional']
profile['segment'] = labels_biz

print('\nProfile tung phan khuc:')
print(profile.to_string())

# Map nhãn ngược về dataframe
seg_map = profile['segment'].to_dict()
rfm_clean['segment'] = rfm_clean['cluster'].map(seg_map)

## 1.5 Trực quan hóa phân khúc

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
SEG_COLORS = {'Champion': '#2196F3', 'Loyal': '#4CAF50',
              'At Risk': '#FF9800', 'New/Occasional': '#E91E63'}
color_list = [SEG_COLORS[s] for s in rfm_clean['segment']]

# Recency vs Monetary
axes[0].scatter(rfm_clean['recency_days'], rfm_clean['monetary'],
                c=color_list, alpha=0.3, s=5)
axes[0].set_xlabel('Recency (days)')
axes[0].set_ylabel('Monetary (BRL)')
axes[0].set_title('Recency vs Monetary')

# Frequency vs Monetary
axes[1].scatter(rfm_clean['frequency'], rfm_clean['monetary'],
                c=color_list, alpha=0.3, s=5)
axes[1].set_xlabel('Frequency (orders)')
axes[1].set_ylabel('Monetary (BRL)')
axes[1].set_title('Frequency vs Monetary')

# Pie chart phân bố
seg_counts = rfm_clean['segment'].value_counts()
axes[2].pie(seg_counts, labels=seg_counts.index,
            colors=[SEG_COLORS[s] for s in seg_counts.index],
            autopct='%1.1f%%', startangle=140)
axes[2].set_title('Phan bo phan khuc khach hang')

# Legend chung
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=c, label=s) for s, c in SEG_COLORS.items()]
fig.legend(handles=legend_els, loc='lower center', ncol=4, fontsize=10, framealpha=0.9)

plt.suptitle('K-Means Customer Segmentation — RFM', fontweight='bold', fontsize=13)
plt.tight_layout(rect=[0, 0.07, 1, 1])
plt.savefig('../data/processed/kmeans_segments.png', bbox_inches='tight')
plt.show()

## 1.6 Tóm tắt kết quả Clustering

In [ ]:
summary = rfm_clean.groupby('segment').agg(
    so_luong=('customer_unique_id', 'count'),
    recency_tb=('recency_days', 'mean'),
    frequency_tb=('frequency', 'mean'),
    monetary_tb=('monetary', 'mean'),
    monetary_total=('monetary', 'sum')
).round(2)
summary['pct_khach'] = (summary['so_luong'] / summary['so_luong'].sum() * 100).round(1)
summary['pct_doanh_thu'] = (summary['monetary_total'] / summary['monetary_total'].sum() * 100).round(1)
summary = summary.sort_values('monetary_tb', ascending=False)

print('=== KET QUA PHAN KHUC KHACH HANG ===')
print(summary[['so_luong','pct_khach','recency_tb','frequency_tb',
               'monetary_tb','pct_doanh_thu']].to_string())

# Bar chart doanh thu theo segment
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(summary.index, summary['pct_doanh_thu'],
              color=[SEG_COLORS[s] for s in summary.index], edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylabel('% Dong gop doanh thu')
ax.set_title('Dong gop Doanh Thu theo Phan Khuc Khach Hang', fontweight='bold')
ax.set_ylim(0, summary['pct_doanh_thu'].max() * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/kmeans_revenue_by_segment.png', bbox_inches='tight')
plt.show()

---
# PHẦN 2: LINEAR REGRESSION — Dự đoán Review Score

**Mục tiêu:** Tìm hiểu các yếu tố ảnh hưởng đến sự hài lòng của khách hàng (`review_score`).

**Features sử dụng:**
- `delivery_days`, `delay_days`, `is_late`
- `price`, `freight_value`, `payment_installments`
- `purchase_month`, `purchase_hour`, `same_state`
- `product_weight_g`, `product_volume_cm3`

## 2.1 Chuẩn bị dữ liệu Regression

In [ ]:
df_reg = pd.read_sql_query("""
    SELECT
        review_score,
        delivery_days,
        delay_days,
        is_late,
        price,
        freight_value,
        payment_installments,
        purchase_month,
        purchase_hour,
        same_state,
        product_weight_g,
        product_volume_cm3
    FROM view_order_clean
    WHERE review_score IS NOT NULL
      AND delivery_days IS NOT NULL
""", conn)

print(f'Du lieu Regression: {len(df_reg):,} dong')
print(f'Missing values:\n{df_reg.isnull().sum()[df_reg.isnull().sum()>0]}')

df_reg = df_reg.dropna()
print(f'\nSau khi bo NA: {len(df_reg):,} dong')
print(f"\nPhan bo review_score:\n{df_reg['review_score'].value_counts().sort_index()}")

## 2.2 Phân tích tương quan với review_score

In [ ]:
FEATURES = ['delivery_days', 'delay_days', 'is_late', 'price',
            'freight_value', 'payment_installments', 'purchase_month',
            'purchase_hour', 'same_state', 'product_weight_g', 'product_volume_cm3']
TARGET = 'review_score'

corr = df_reg[FEATURES + [TARGET]].corr()[[TARGET]].drop(TARGET)
corr = corr.sort_values(TARGET)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#E53935' if v < 0 else '#2196F3' for v in corr[TARGET]]
bars = ax.barh(corr.index, corr[TARGET], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation voi review_score')
ax.set_title('Tuong quan giua cac dac trung va Review Score', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, corr[TARGET]):
    ax.text(val + (0.002 if val >= 0 else -0.002), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/regression_correlation.png', bbox_inches='tight')
plt.show()

## 2.3 Huấn luyện Linear Regression

In [ ]:
X = df_reg[FEATURES]
y = df_reg[TARGET]

# Chia train/test 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Chuẩn hóa
scaler_reg = StandardScaler()
X_train_s = scaler_reg.fit_transform(X_train)
X_test_s  = scaler_reg.transform(X_test)

# Huấn luyện
lr = LinearRegression()
lr.fit(X_train_s, y_train)

# Đánh giá
y_pred = lr.predict(X_test_s)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# Cross-validation 5-fold
cv_r2 = cross_val_score(LinearRegression(), X_train_s, y_train,
                         cv=5, scoring='r2')

print('=== KET QUA LINEAR REGRESSION ===')
print(f'  Train size : {len(X_train):,}  |  Test size: {len(X_test):,}')
print(f'  R²  (test) : {r2:.4f}')
print(f'  MAE (test) : {mae:.4f}')
print(f'  RMSE(test) : {rmse:.4f}')
print(f'  R² 5-fold CV: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')

## 2.4 Phân tích hệ số hồi quy (Feature Importance)

In [ ]:
coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coefficient': lr.coef_
}).sort_values('coefficient')

fig, ax = plt.subplots(figsize=(9, 5))
colors_coef = ['#E53935' if v < 0 else '#2196F3' for v in coef_df['coefficient']]
bars = ax.barh(coef_df['feature'], coef_df['coefficient'],
               color=colors_coef, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('He so hoi quy (chuan hoa)')
ax.set_title(f'He so Linear Regression | R² = {r2:.4f}', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, coef_df['coefficient']):
    ax.text(val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/regression_coefficients.png', bbox_inches='tight')
plt.show()

print('\nTop yeu to ANH HUONG TICH CUC den review_score:')
print(coef_df[coef_df['coefficient']>0].sort_values('coefficient',ascending=False)[['feature','coefficient']].to_string(index=False))
print('\nTop yeu to ANH HUONG TIEU CUC den review_score:')
print(coef_df[coef_df['coefficient']<0].sort_values('coefficient')[['feature','coefficient']].to_string(index=False))

## 2.5 Biểu đồ Actual vs Predicted & Residuals

In [ ]:
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted (jitter để thấy mật độ)
jitter = np.random.normal(0, 0.1, len(y_test))
axes[0].scatter(y_pred, y_test.values + jitter,
                alpha=0.08, s=4, color=PALETTE[0])
mn, mx = y_pred.min(), y_pred.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Predicted review_score')
axes[0].set_ylabel('Actual review_score')
axes[0].set_title(f'Actual vs Predicted | R² = {r2:.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual distribution
axes[1].hist(residuals, bins=40, color=PALETTE[1], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Phan phoi Residuals | MAE={mae:.4f}, RMSE={rmse:.4f}')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Danh gia mo hinh Linear Regression', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/regression_evaluation.png', bbox_inches='tight')
plt.show()

---
# PHẦN 3: TỔNG KẾT & NHẬN XÉT

In [ ]:
print('=' * 60)
print('  KET QUA MO HINH HOC MAY — OLIST E-COMMERCE')
print('=' * 60)

print('\n--- K-MEANS CLUSTERING (RFM) ---')
print(f'  So cum (k)        : {K_FINAL}')
print(f'  Silhouette Score  : {sil:.4f}')
print(f'  Inertia           : {km_final.inertia_:,.0f}')
print()
for seg, row in summary.iterrows():
    print(f'  [{seg:16s}] {row["so_luong"]:>6,} KH '
          f'| R={row["recency_tb"]:>5.0f}d '
          f'| F={row["frequency_tb"]:>4.2f} '
          f'| M=R${row["monetary_tb"]:>7.2f} '
          f'| {row["pct_doanh_thu"]}% DT')

print('\n--- LINEAR REGRESSION (predict review_score) ---')
print(f'  R² (test)         : {r2:.4f}')
print(f'  MAE               : {mae:.4f}')
print(f'  RMSE              : {rmse:.4f}')
print(f'  R² 5-fold CV      : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print()

top_neg = coef_df[coef_df['coefficient']<0].iloc[0]
top_pos = coef_df[coef_df['coefficient']>0].iloc[-1]
print(f'  Yeu to anh huong tieu cuc nhat : {top_neg["feature"]} (coef={top_neg["coefficient"]:.3f})')
print(f'  Yeu to anh huong tich cuc nhat : {top_pos["feature"]} (coef={top_pos["coefficient"]:.3f})')
print('=' * 60)

conn.close()
print('\nDong ket noi DB.')